In [1]:
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd

In [2]:
centres = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2025-26",
    season_type_all_star="Regular Season",
    per_mode_detailed="Totals",
    player_position_abbreviation_nullable="C",
    timeout=60
).get_data_frames()[0]

# Keep only the columns we need
centres_3pt = centres[["PLAYER_NAME", "TEAM_ABBREVIATION", "GP", "FG3A", "FG3M"]].copy()

# Misses aren't a column, so calculate them
centres_3pt["FG3_MISSED"] = centres_3pt["FG3A"] - centres_3pt["FG3M"]

# Sort by attempts, most first
centres_3pt = centres_3pt.sort_values("FG3A", ascending=False)
pd.set_option("display.max_rows", None)
centres_3pt

,PLAYER_NAME,TEAM_ABBREVIATION,GP,FG3A,FG3M,FG3_MISSED
76,Naz Reid,MIN,77,448,162,286
7,Bam Adebayo,MIA,73,400,127,273
82,Noah Clowney,BKN,66,392,129,263
74,Myles Turner,MIL,71,384,147,237
85,Onyeka Okongwu,ATL,74,383,144,239
43,Jay Huff,IND,82,373,119,254
101,Victor Wembanyama,SAS,64,350,122,228
50,Julius Randle,MIN,79,346,109,237
62,Lauri Markkanen,UTA,42,324,115,209
10,Brook Lopez,LAC,75,317,114,203


In [3]:
import time
import os

os.makedirs("data/raw", exist_ok=True)

seasons = [f"{y}-{str(y + 1)[-2:]}" for y in range(2006, 2026)]
all_seasons = []

for season in seasons:
    try:
        df = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            season_type_all_star="Regular Season",
            per_mode_detailed="Totals",
            player_position_abbreviation_nullable="C",
            timeout=60
        ).get_data_frames()[0]
        df["SEASON"] = season
        df.to_csv(f"data/raw/centres_{season}.csv", index=False)
        all_seasons.append(df)
        print(f"{season}: {len(df)} centres")
    except Exception as e:
        print(f"{season}: FAILED ({e})")
    time.sleep(2)

centres_all = pd.concat(all_seasons, ignore_index=True)
centres_all.to_csv("data/centres_2006_2026.csv", index=False)

# Season totals
season_3pt = centres_all.groupby("SEASON")[["FG3A", "FG3M"]].sum()
season_3pt["FG3_MISSED"] = season_3pt["FG3A"] - season_3pt["FG3M"]
season_3pt["FG3_PCT"] = (season_3pt["FG3M"] / season_3pt["FG3A"]).round(3)
season_3pt


2006-07: 109 centres
2007-08: 118 centres
2008-09: 116 centres
2009-10: 109 centres
2010-11: 103 centres
2011-12: 120 centres
2012-13: 112 centres
2013-14: 114 centres
2014-15: 113 centres
2015-16: 108 centres
2016-17: 101 centres
2017-18: 118 centres
2018-19: 100 centres
2019-20: 103 centres
2020-21: 111 centres
2021-22: 112 centres
2022-23: 105 centres
2023-24: 101 centres
2024-25: 107 centres
2025-26: 111 centres


,FG3A,FG3M,FG3_MISSED,FG3_PCT
SEASON,,,,
2006-07,1446,499,947,0.345
2007-08,2231,745,1486,0.334
2008-09,2271,868,1403,0.382
2009-10,2503,863,1640,0.345
2010-11,1512,553,959,0.366
2011-12,1606,521,1085,0.324
2012-13,1459,438,1021,0.300
2013-14,2803,942,1861,0.336
2014-15,1807,581,1226,0.322
